<a href="https://colab.research.google.com/github/hadasecohen/streaming-vae-anomaly-detection/blob/main/scripts/compile_test_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compile src/ for Colab (Cython)

**Purpose:** compile `src/` into `.so` extension modules on Colab's own Linux/Python runtime, and download the result as `compiled_src.zip`. This is the standard way to produce a compiled build for the public/production repo — paste the zip's `src/` over the public repo's `src/` to release a new version with no readable `.py` source.

Colab's runtime is Linux/x86_64 with `gcc` preinstalled, so no compiler setup is needed (unlike a local Windows machine, which needs MSVC Build Tools installed separately — see `scripts/local/compile_src.py`'s docstring for that path).

### Step 1 — Clone this (private) repo

Needs a GitHub token with read access: add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) and enable **Notebook access** for it.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"
REPO_DIR = Path("/content/repo_compile")

from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError(
        "The Colab secret GITHUB_TOKEN is unavailable. "
        "Open the key icon in the left sidebar, add the token, "
        "and enable Notebook access."
    )

if not REPO_DIR.exists():
    # Use an askpass helper so the token is not stored in the Git remote URL.
    askpass = Path("/content/git_askpass_compile.sh")
    askpass_body = (
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    )
    askpass.write_text(askpass_body)
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True,
            env=env,
        )
    finally:
        askpass.unlink(missing_ok=True)
else:
    # Already cloned in this session — pull latest so a stale checkout
    # doesn't silently compile old source.
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

### Step 2 — Install Cython

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cython"], check=True)

import Cython
print("Cython version:", Cython.__version__)
print("Python version:", sys.version)

### Step 3 — Compile, verify, clean, and zip — all in one command

`scripts/local/compile_src.py` runs the full pipeline in order: copy `src/` → compile every file with Cython (`annotation_typing=False`, since this codebase's type hints aren't reliable contracts everywhere) → verify the compiled module count matches the source file count → delete the original `.py` source (so it doesn't shadow the compiled modules on import) → delete intermediate `.c` files → zip the result. This is the only command you need to run to produce a fresh `compiled_src.zip`.

In [ ]:
!python scripts/local/compile_src.py --out build/compiled_src 2>&1 | tail -80

### Step 4 — Download `compiled_src.zip`

In [ ]:
from google.colab import files
files.download("build/compiled_src.zip")